## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the platform.

In [1]:
import digitalhub as dh
PROJECT_NAME = "cancer-classifier"
proj = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## 2. Execution

Fetch the "hello-world" operation in the project. 

In [4]:
func_mlflow_train_model = proj.get_function("cancer-classifier-train") 

Run the function as shown below. This will trigger the model training job on the platform and the model 'model-mlflow' get logged on to the platform.

In [ ]:
func_mlflow_train_model.run(action='job')

Now, one can fetch the trained model saved in project.


In [8]:
model = proj.get_model("cancer_classifier")

Note: Alternatively, using the Core Management UI, one can navigate to 'Model' menu to see the generated model.

### Serve

The 'serve' action deploys MLflow ML models as services on Kubernetes. A Task is created by calling run() on the Function; task parameters are passed through that call.

In [15]:
serve_func = proj.new_function(
    name="cancer-classifier-service",
    kind="sklearnserve",
    model_name=model.name,
    path=model.key,
)

In [ ]:
serve_run = serve_func.run("serve", wait=True)

#### Test the model

Let's test our deployed MLflow model by making a prediction request with sample Iris data:

In [17]:
import numpy as np

# Generate sample data for prediction
data = np.random.rand(2, 30).tolist()
json_payload = {
    "inputs": [{"name": "input-0", "shape": [2, 30], "datatype": "FP32", "data": data}]
}

# Make prediction
result = serve_run.refresh().invoke(model_name=model.name, json=json_payload).json()
print("Prediction result:")
print(result)

Prediction result:
{'model_name': 'cancer_classifier', 'id': '72ea7ae0-ecf0-42d3-96a2-497dc6449ae9', 'parameters': {}, 'outputs': [{'name': 'predict', 'shape': [2, 1], 'datatype': 'INT64', 'parameters': {'content_type': 'np'}, 'data': [1, 1]}]}
